# 06 — 对比分析与可视化

汇总所有实验配置的结果，生成对比图表。

**控制变量法 — 2 个独立优化维度，3 个实验点：**

| 配置 | 优化维度 | Cache 管理 | Cache 精度 | 框架 |
|------|----------|------------|------------|------|
| GQA Only (Baseline) | 无 | DynamicCache (连续) | FP16 | PyTorch |
| GQA + PagedAttn | 内存管理 | PagedAttention (分页) | FP16 | vLLM |
| GQA + KIVI | 数值压缩 | DynamicCache (连续) | 2-bit | PyTorch + KIVI CUDA |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_style('whitegrid')
RESULTS = Path('../results')

In [ ]:
# 加载所有 CSV
dfs = []
for csv_name, config_name in [
    ('baseline_gqa.csv', 'GQA Only'),
    ('gqa_paged_vllm.csv', 'GQA+PagedAttn(vLLM)'),
    ('gqa_kivi.csv', 'GQA+KIVI'),
]:
    path = RESULTS / csv_name
    if path.exists():
        df = pd.read_csv(path)
        df['config'] = config_name
        dfs.append(df)
        print(f"✓ {csv_name}: {len(df)} rows")
    else:
        print(f"✗ {csv_name}: not found")

if dfs:
    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal: {len(combined)} rows across {combined['config'].nunique()} configs")
else:
    print("\n⚠️  没有结果文件！请先运行 notebooks 02-05")

In [ ]:
# 汇总表
if dfs:
    summary = combined.groupby('config').agg({
        'ttft_ms': ['mean', 'std'],
        'tpot_ms': ['mean', 'std'],
        'peak_memory_mb': 'max',
        'kv_cache_memory_mb': 'mean',
    }).round(2)
    
    # 如果有 memory_fragmentation
    if 'memory_fragmentation' in combined.columns:
        summary[('memory_fragmentation', 'mean')] = combined.groupby('config')['memory_fragmentation'].mean().round(3)
    
    print(summary.to_string())

---
## TTFT 对比

TTFT 主要受 Memory Bandwidth 影响：预填充时需要大量读取权重。
KIVI 量化不影响 prefill（KV 此时尚未量化）。

In [ ]:
if dfs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # TTFT
    sns.boxplot(data=combined, x='config', y='ttft_ms', ax=axes[0],
                palette='Blues_d')
    axes[0].set_title('TTFT (Time To First Token)', fontsize=13)
    axes[0].set_ylabel('ms')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=30)
    
    # TPOT
    sns.boxplot(data=combined, x='config', y='tpot_ms', ax=axes[1],
                palette='Oranges_d')
    axes[1].set_title('TPOT (Time Per Output Token)', fontsize=13)
    axes[1].set_ylabel('ms')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.savefig('../results/ttft_tpot_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 内存使用对比

关键：在 Jetson 10-12 GB 预算下，各配置能处理多长的上下文？

In [ ]:
if dfs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Peak memory
    mem_summary = combined.groupby('config')['peak_memory_mb'].max().sort_values()
    colors = sns.color_palette('viridis', n_colors=len(mem_summary))
    mem_summary.plot(kind='barh', ax=axes[0], color=colors)
    axes[0].axvline(x=11 * 1024, color='red', linestyle='--',
                    label='Jetson budget (11 GB)')
    axes[0].set_xlabel('Peak Memory (MB)')
    axes[0].set_title('Peak GPU Memory Usage')
    axes[0].legend()
    
    # KV Cache memory
    if 'kv_cache_memory_mb' in combined.columns:
        kv_summary = combined.groupby('config')['kv_cache_memory_mb'].mean().sort_values()
        kv_summary.plot(kind='barh', ax=axes[1], color=colors)
        axes[1].set_xlabel('KV Cache Memory (MB)')
        axes[1].set_title('Average KV Cache Memory')
    
    plt.tight_layout()
    plt.savefig('../results/memory_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## TPOT vs Input Length (KIVI 效果验证)

KIVI 的核心价值：随着上下文变长，TPOT 增长**极其平缓**（显存搬运量被压缩了）。

In [ ]:
if dfs and 'num_input_tokens' in combined.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for config in combined['config'].unique():
        subset = combined[combined['config'] == config].sort_values('num_input_tokens')
        ax.plot(subset['num_input_tokens'], subset['tpot_ms'],
                'o-', label=config, alpha=0.8, markersize=4)
    
    ax.set_xlabel('Input Token Length')
    ax.set_ylabel('TPOT (ms)')
    ax.set_title('TPOT vs Input Length — KIVI 让显存曲线变平缓')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/tpot_vs_length.png', dpi=150)
    plt.show()

---
## PPL 质量退化对比

PPL 增加意味着模型预测能力下降。在医疗场景中，这关乎推理逻辑的连贯性。

In [ ]:
# 加载 PPL 数据（双来源：JSON 汇总 + 各 notebook 的 CSV）
ppl_path = RESULTS / 'ppl_results.json'
ppl_csvs = [
    ('ppl_gqa_only.csv', 'GQA Only'),
    ('ppl_gqa_paged.csv', 'GQA+PagedAttn'),
    ('ppl_gqa_kivi.csv', 'GQA+KIVI'),
]

# 优先用 CSV（更可靠，每个 notebook 独立输出）
ppl_rows = []
for csv_name, config_name in ppl_csvs:
    p = RESULTS / csv_name
    if p.exists():
        row = pd.read_csv(p).iloc[0]
        ppl_rows.append({'config': config_name, 'ppl': row['ppl'],
                         'avg_loss': row['avg_loss'], 'num_tokens': int(row['num_tokens'])})
        print(f"✓ {csv_name}")
    else:
        print(f"✗ {csv_name}: not found")

# 兜底从 JSON 补漏
if ppl_path.exists():
    with open(ppl_path) as f:
        ppl_json = json.load(f)
    loaded_configs = {r['config'] for r in ppl_rows}
    json_map = {'GQA_only': 'GQA Only', 'GQA+PagedAttn': 'GQA+PagedAttn',
                'GQA+KIVI': 'GQA+KIVI', 'GQA+Paged+KIVI': 'All Combined'}
                'GQA+KIVI': 'GQA+KIVI'}
        display_name = json_map.get(k, k)
        if display_name not in loaded_configs:
        if display_name not in loaded_configs and display_name in json_map.values():
                             'avg_loss': v['avg_loss'], 'num_tokens': v['num_tokens']})
            print(f"  (from JSON) {k}")

if ppl_rows:
    ppl_df = pd.DataFrame(ppl_rows)
    print(f"\nPPL data: {len(ppl_df)} configs")
    print(ppl_df.to_string(index=False))
else:
    ppl_df = None
    print("\n⚠️ 没有 PPL 数据。请先运行 notebooks 02-05")
    print("\n⚠️ 没有 PPL 数据。请先运行 notebooks 02-04")
# 可视化
if ppl_df is not None and len(ppl_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = sns.color_palette('Set2', len(ppl_df))
    bars = ax.bar(ppl_df['config'], ppl_df['ppl'], color=colors, edgecolor='black')
    ax.set_ylabel('Perplexity (PPL)')
    ax.set_title('Quality Assessment: Perplexity Comparison')

    # 5% 退化线
    baseline_row = ppl_df[ppl_df['config'] == 'GQA Only']
    if not baseline_row.empty:
        threshold = baseline_row.iloc[0]['ppl'] * 1.05
        ax.axhline(y=threshold, color='red', linestyle='--',
                   label=f'5% degradation threshold ({threshold:.1f})')
        ax.legend()

    for bar, ppl_val in zip(bars, ppl_df['ppl']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{ppl_val:.2f}', ha='center', fontsize=10)

    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.savefig('../results/ppl_comparison.png', dpi=150)
    plt.show()

---
## 内存碎片对比

碎片 = 1 - allocated/reserved。PagedAttention 应该明显降低碎片率。

In [ ]:
if dfs and 'memory_fragmentation' in combined.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=combined, x='config', y='memory_fragmentation',
                palette='coolwarm', ax=ax)
    ax.set_ylabel('Memory Fragmentation (1 - alloc/reserved)')
    ax.set_title('Memory Fragmentation Comparison')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig('../results/fragmentation_comparison.png', dpi=150)
    plt.show()

---
## 综合结论导出

In [ ]:
if dfs:
    # 最终汇总 CSV（性能）
    combined.to_csv('../results/all_experiments_combined.csv', index=False)
    
    # 性能摘要表
    final_summary = combined.groupby('config').agg({
        'ttft_ms': 'mean',
        'tpot_ms': 'mean',
        'peak_memory_mb': 'max',
        'kv_cache_memory_mb': 'mean',
    }).round(1)
    
    # 合并 PPL 数据
    if ppl_df is not None:
        ppl_merge = ppl_df.set_index('config')[['ppl']]
        final_summary = final_summary.join(ppl_merge, how='left')
    
    # 相对于 baseline 的变化
    if 'GQA Only' in final_summary.index:
        baseline = final_summary.loc['GQA Only']
        for col in final_summary.columns:
            if pd.api.types.is_numeric_dtype(final_summary[col]):
                final_summary[f'{col}_vs_baseline_%'] = (
                    (final_summary[col] - baseline[col]) / baseline[col] * 100
                ).round(1)
    
    final_summary.to_csv('../results/final_summary.csv')
    print("✓ final_summary.csv 已保存\n")
    print(final_summary.to_string())
else:
    print("⚠️ 没有性能数据")